In [23]:
import warnings
import pandas as pd
import pyreadr
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    root_mean_squared_error,
    accuracy_score,
    classification_report,
    f1_score,
    confusion_matrix,
    precision_score,
    recall_score,
    ConfusionMatrixDisplay
 )
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore', category=RuntimeWarning, message='invalid value encountered in reduce')

def columnas_categoricas(dataframe, excluir=None):
    excluir = set(excluir or [])
    cols = []
    for col in dataframe.columns:
        if col in excluir:
            continue
        dtype = dataframe[col].dtype
        if (
            pd.api.types.is_object_dtype(dtype)
            or pd.api.types.is_string_dtype(dtype)
            or isinstance(dtype, pd.CategoricalDtype)
            or pd.api.types.is_bool_dtype(dtype)
        ):
            cols.append(col)
    return cols

resultado = pyreadr.read_r('listings.RData')
df = list(resultado.values())[0]
print(f"Dimensiones iniciales del dataset: {df.shape}")

numericas = df.select_dtypes(include=['number']).columns.tolist()
categoricas = columnas_categoricas(df)

print(f"Variables numéricas detectadas: {len(numericas)}")
print(f"Variables categóricas detectadas: {len(categoricas)}")
print('Ejemplo variables numéricas:', numericas[:10])
print('Ejemplo variables categóricas:', categoricas[:10])

limpio = (
    df['price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
 )
df['price'] = pd.to_numeric(limpio, errors='coerce')
print('Conversión de price completada.')

Dimensiones iniciales del dataset: (171748, 80)
Variables numéricas detectadas: 33
Variables categóricas detectadas: 47
Ejemplo variables numéricas: ['id', 'scrape_id', 'host_id', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'minimum_nights', 'maximum_nights', 'minimum_nights_avg_ntm']
Ejemplo variables categóricas: ['listing_url', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_url', 'host_name', 'host_since']
Conversión de price completada.


In [24]:
df = df.replace([np.inf, -np.inf], np.nan)

faltantes = df.isnull().sum()
faltantes = faltantes[faltantes > 0].sort_values(ascending=False)

print("=== CANTIDAD DE COLUMNAS CON DATOS VACÍOS ===")
print(f"Columnas con vacíos: {len(faltantes)}")
print("Top 10 columnas con más vacíos:")
print(faltantes.head(10))
print("\n" + "=" * 40 + "\n")

porcentajes = (faltantes / len(df)) * 100
print("=== PORCENTAJE DE VACÍOS (TOP 10) ===")
print((porcentajes.head(10).round(2)).astype(str) + " %")

df = df.dropna(subset=['price'])
df = df.dropna(axis=1, how='all')

numericas = df.select_dtypes(include=['number']).columns.tolist()
if 'price' in numericas:
    numericas.remove('price')

categoricas = columnas_categoricas(df, excluir=['price'])

imputador_num = SimpleImputer(strategy='median')
imputador_cat = SimpleImputer(strategy='constant', fill_value='Sin Dato')

df[numericas] = imputador_num.fit_transform(df[numericas])
df[categoricas] = imputador_cat.fit_transform(df[categoricas])

print('Filas sin precio eliminadas y faltantes imputados.')
print('Total de nulos restantes:', df.isnull().sum().sum())
print('Dimensiones luego de limpieza:', df.shape)

=== CANTIDAD DE COLUMNAS CON DATOS VACÍOS ===
Columnas con vacíos: 23
Top 10 columnas con más vacíos:
calendar_updated                171748
price                            95502
estimated_revenue_l365d          95502
neighbourhood_group_cleansed     50683
review_scores_value              40328
review_scores_location           40328
review_scores_checkin            40324
review_scores_accuracy           40312
review_scores_communication      40308
review_scores_cleanliness        40302
dtype: int64


=== PORCENTAJE DE VACÍOS (TOP 10) ===
calendar_updated                100.0 %
price                           55.61 %
estimated_revenue_l365d         55.61 %
neighbourhood_group_cleansed    29.51 %
review_scores_value             23.48 %
review_scores_location          23.48 %
review_scores_checkin           23.48 %
review_scores_accuracy          23.47 %
review_scores_communication     23.47 %
review_scores_cleanliness       23.47 %
dtype: str
Filas sin precio eliminadas y faltantes impu

# Laboratorio 7.
# Regresion Logistica

Vianka Castro - 23201
Ricardo Godinez -23247
Felipe Aguilar -23195

## 1. Cree una variable dicotómica por cada una de las categorías de la variable respuesta categórica que creó en hojas anteriores. Debería tener 3 variables dicotómicas (valores 0 y 1) una que diga si la vivienda es cara o no, media o no, económica o no.

In [25]:
q1, q2 = df['price'].astype(float).quantile([1/3, 2/3])

import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn.impute')

df['categoria_precio'] = pd.cut(
    df['price'].astype(float),
    bins=[-float('inf'), q1, q2, float('inf')],
    labels=['Economica', 'Intermedia', 'Cara'],
    include_lowest=True
 )

df['es_economica'] = (df['categoria_precio'] == 'Economica').astype(int)
df['es_intermedia'] = (df['categoria_precio'] == 'Intermedia').astype(int)
df['es_cara'] = (df['categoria_precio'] == 'Cara').astype(int)

print(df[['categoria_precio', 'es_economica', 'es_intermedia', 'es_cara']].head())

  categoria_precio  es_economica  es_intermedia  es_cara
0        Economica             1              0        0
1       Intermedia             0              1        0
2        Economica             1              0        0
3       Intermedia             0              1        0
4        Economica             1              0        0


Se deben crear variables dicotómicas (0 y 1) porque la Regresión Logística es un modelo matemático que solo entiende números, no palabras.

Al igual que otros modelos basados en matemáticas, la Regresión Logística funciona multiplicando los datos de entrada por ciertos pesos (coeficientes) para calcular una probabilidad. Como no puede multiplicar un texto como "Económica" o "Cara", la conversión a variables dicotómicas (conocida como One-Hot Encoding) nos permite representar la presencia (1) o ausencia (0) de una característica.

## 2. Use los mismos conjuntos de entrenamiento y prueba que utilizó en las hojas anteriores.

In [26]:
# 1. ELIMINAR NULOS Y DEFINIR 'X' e 'Y'
# Filtramos las filas donde la categoría no sea nula
filas_sanas = df['categoria_precio'].notna()
df_limpio = df[filas_sanas].copy()

# 'X' va a ser todo EXCEPTO el precio original y las categorías que creamos
X = df_limpio.drop(columns=['price', 'categoria_precio', 'es_economica', 'es_intermedia', 'es_cara'])
X = X.replace([np.inf, -np.inf], np.nan)

# 'Y_strata' solo sirve para asegurar que la partición sea la misma de KNN
Y_strata = df_limpio['categoria_precio']

# 'Y_dicotomicas' almacena las tres columnas que vamos a predecir
Y_dicotomicas = df_limpio[['es_economica', 'es_intermedia', 'es_cara']]

# Detección de tipos para el preprocesador (igual al lab anterior)
numericas = X.select_dtypes(include=['number']).columns.tolist()
categoricas_crudas = X.select_dtypes(include=['object', 'category']).columns.tolist() 
categoricas = [col for col in categoricas_crudas if X[col].nunique(dropna=True) < 50]
X[categoricas] = X[categoricas].astype(str)

# 2. SEPARACIÓN DE DATOS (El Split idéntico)
X_train_clf, X_test_clf, Y_train_dico, Y_test_dico = train_test_split(
    X, Y_dicotomicas, test_size=0.3, random_state=42, stratify=Y_strata
)


preprocesador_clf = ColumnTransformer(
    transformers=[
        ('numeros', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())  # ¡Muy importante escalarlo en Reg. Logística!
        ]), numericas),
        ('textos', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categoricas)
    ]
)



C:\Users\richi\AppData\Local\Temp\ipykernel_12912\570162800.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categoricas_crudas = X.select_dtypes(include=['object', 'category']).columns.tolist()


Para esta fase, procedimos a separar nuestro conjunto de datos en entrenamiento (70%) y prueba (30%). Con el objetivo de garantizar una comparación rigurosa y justa frente a los mocelos construidos en laboratorios anteriores, replicamos exactamente la misma semilla de aleatoriedad (random_state=42) y utilizamos la misma variable objetivo multiclase para aplicar el muestreo estratificado (stratify). Esto nos asegura que tanto los modelos anteriores como los futuros modelos de Regresión Logística serán entrenados y evaluados utilizando el 100% de los mismos registros, permitiéndonos contrastar su rendimiento de manera objetiva sin sesgos en la distribución de los datos

## 3. Elabore un modelo de regresión logística para conocer si el precio de una vivienda es cara o no 
utilizando el conjunto de entrenamiento y explique los resultados a los que llega. El
experimento debe ser reproducible por lo que debe fijar que los conjuntos de
entrenamiento y prueba sean los mismos siempre que se ejecute el código. Use validación
cruzada.

In [ ]:
print("=== MODELO (ES_CARA) CON VALIDACIÓN CRUZADA ===")

Y_train_cara = Y_train_dico['es_cara']
Y_test_cara = Y_test_dico['es_cara']

pipeline_log_cara = Pipeline(steps=[
    ('preprocesador', preprocesador_clf),
    ('clasificador', LogisticRegression(max_iter=1000, random_state=42))
])

print("\nRealizando Validación Cruzada (5 particiones en el set de entrenamiento)...")
cv_scores = cross_val_score(pipeline_log_cara, X_train_clf, Y_train_cara, cv=5, scoring='accuracy')
print(f"Precisión en cada partición: {cv_scores}")

print(f"Exactitud Promedio (CV): {cv_scores.mean():.4f} +/- {cv_scores.std():.4f} (Desviación)")

pipeline_log_cara.fit(X_train_clf, Y_train_cara)
pred_cara = pipeline_log_cara.predict(X_test_clf)
print("\n--- RESULTADOS EN EL CONJUNTO DE PRUEBA ---")
print(f"Exactitud (Accuracy): {accuracy_score(Y_test_cara, pred_cara):.4f}")
print(f"Precisión (Precision): {precision_score(Y_test_cara, pred_cara, zero_division=0):.4f}")
print(f"Sensibilidad (Recall): {recall_score(Y_test_cara, pred_cara, zero_division=0):.4f}")
print(f"F1-Score: {f1_score(Y_test_cara, pred_cara, zero_division=0):.4f}")

=== PASO 3: MODELO (ES_CARA) CON VALIDACIÓN CRUZADA ===

Realizando Validación Cruzada (5 particiones en el set de entrenamiento)...
Precisión en cada partición: [0.8206089  0.81386417 0.82415215 0.81815627 0.82396477]
Exactitud Promedio (CV): 0.8201 +/- 0.0039 (Desviación)

--- RESULTADOS EN EL CONJUNTO DE PRUEBA (Nunca antes visto) ---
Exactitud (Accuracy): 0.8149
Precisión (Precision): 0.7574
Sensibilidad (Recall): 0.6540
F1-Score: 0.7019


Para validar la estabilidad del modelo y garantizar que sus resultados no fuesen producto del azar, se aplicó una Validación Cruzada de 5 particiones (K-Fold) sobre el conjunto de entrenamiento. Los resultados demostraron una consistencia sobresaliente, con una exactitud promedio del 82.01% y una desviación estándar minúscula (± 0.0039). Esto nos asegura que el modelo no sufre de sobreajuste (overfitting) y es capaz de aprender patrones fiables independientemente del bloque de datos evaluado.

Al enfrentar el modelo a los datos de prueba, la exactitud obtenida fue del 81.49%, comprobando que el modelo generaliza de forma excelente, mimetizando el rendimiento de la validación cruzada. Adicionalmente, cuenta con una Precisión del 75.74% y una Sensibilidad (Recall) del 65.40%, consolidando un F1-Score general de 0.7019. Estos indicadores concluyen que la Regresión Logística es considerablemente eficaz distinguiendo las viviendas de alto valor del resto del mercado, siendo sus predicciones afirmativas ('es cara') confiables 3 de cada 4 veces.